In [0]:
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import Window
import pyspark.sql.functions as F

CATALOG_SCHEMA  = "teste_koin.default."
BRONZE_TABLE    = "bronze_orders"
SILVER_TABLE    = "silver_orders"
MERGE_KEY       = "order_id"
DEDUP_ORDER_COL = "order_date"

def assert_columns_exist(df, required_cols: list[str], stage: str) -> None:
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{stage}] Colunas ausentes na tabela: {missing}")

df_bronze = spark.table(CATALOG_SCHEMA + BRONZE_TABLE)

REQUIRED_COLUMNS = [
    "order_id", "customer_id","order_date","amount","payment_method","status",
]

assert_columns_exist(df_bronze, REQUIRED_COLUMNS, "bronze")

df_valid_keys = df_bronze.filter(F.col(MERGE_KEY).isNotNull())

# Deduplicação
# Critério: ordenar por order_date de forma decrescente e manter apenas o primeiro registro de cada order_id
window_spec = Window.partitionBy("order_id").orderBy(F.col(DEDUP_ORDER_COL).desc())

df_deduped = (
    df_valid_keys
    .withColumn("_row_number", F.row_number().over(window_spec))
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

# ATENÇÃO: formatos como dd/MM/yyyy e MM/dd/yyyy são ambíguos seria nescessario validar na origem qual o padrao correto para puxar com exatidao exemplo da data 05/06/2024.
order_date_parsed = F.coalesce(
    F.expr("try_to_date(order_date, 'yyyy-MM-dd')"),
    F.expr("try_to_date(order_date, 'dd/MM/yyyy')"),
    F.expr("try_to_date(order_date, 'yyyy/MM/dd')"),
)

amount_clean = (
    F.when(
        F.col("amount").contains(","),
        F.regexp_replace(
            F.regexp_replace(F.col("amount"), r"\.", ""),
            ",", "."
        )
    )
    .otherwise(F.col("amount"))
)

silver_timestamp = F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")).cast("timestamp")

df_silver_orders = df_deduped.select(
    F.col("order_id").cast("string"),
    F.col("customer_id").cast("string"),
    amount_clean.cast("decimal(15,2)").alias("order_amount"),
    F.col("payment_method").alias("order_payment_method"),
    F.col("status").alias("order_status"),
    order_date_parsed.alias("order_date"),
    F.col("ingestion_date").alias("bronze_at"),
    silver_timestamp.alias("silver_at"),
)

# Prefiri o MERGE ao overwrite completo para:
#   1. Preservar histórico de auditoria da tabela Delta.
#   2. Reduzir custo de I/O em tabelas grandes.
#   3. Evitar janelas sem dados durante a reescrita.

if spark.catalog.tableExists(CATALOG_SCHEMA + SILVER_TABLE):
    delta_table = DeltaTable.forName(spark, CATALOG_SCHEMA + SILVER_TABLE)
    (
        delta_table.alias("target")
        .merge(
            df_silver_orders.alias("source"),
            f"target.{MERGE_KEY} = source.{MERGE_KEY}",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("MERGE concluído com sucesso.")

else:
    (
        df_silver_orders.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(CATALOG_SCHEMA + SILVER_TABLE)
    )
    print("Tabela Silver criada com sucesso.")

In [0]:
%sql
drop table teste_koin.default.silver_orders